# Object frequency bar plot

The notebook calculates a cutoff from the recurring-object frequency curve and then uses that result in the interactive graph.

In [1]:
# %pip install anywidget 

In [2]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

csv_path = Path("object_frequencies.csv")
frequency_data = pd.read_csv(csv_path).sort_values("rank").reset_index(drop=True)

required_columns = {"rank", "object", "frequency"}
missing_columns = required_columns.difference(frequency_data.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print(f"Objects: {len(frequency_data):,}")

Objects: 18,254


In [3]:
# Exclude singletons so their long plateau cannot determine the elbow.
recurring_objects = frequency_data.loc[frequency_data["frequency"] > 1].copy()

# Kneedle-style detection on the normalized recurring-object curve.
normalized_rank = (
    (recurring_objects["rank"] - recurring_objects["rank"].min())
    / (recurring_objects["rank"].max() - recurring_objects["rank"].min())
)
normalized_frequency = (
    (recurring_objects["frequency"] - recurring_objects["frequency"].min())
    / (recurring_objects["frequency"].max() - recurring_objects["frequency"].min())
)
distance_from_endpoint_line = (1 - normalized_rank) - normalized_frequency
knee_index = distance_from_endpoint_line.idxmax()
calculated_cutoff_frequency = int(frequency_data.loc[knee_index, "frequency"])

# Include every object tied at the detected frequency.
calculated_cutoff_rank = int(
    frequency_data.loc[
        frequency_data["frequency"] >= calculated_cutoff_frequency, "rank"
    ].max()
)
calculated_cutoff_percentage = calculated_cutoff_rank / len(frequency_data) * 100

frequency_data["cutoff_group"] = frequency_data["rank"].le(
    calculated_cutoff_rank
).map({True: "Before cutoff", False: "After cutoff"})

print(f"Calculated cutoff: after rank {calculated_cutoff_rank:,}")
print(f"Minimum frequency before cutoff: {calculated_cutoff_frequency}")
print(f"Objects before cutoff: {calculated_cutoff_percentage:.2f}%")

Calculated cutoff: after rank 115
Minimum frequency before cutoff: 15
Objects before cutoff: 0.63%


In [4]:
def plot_object_frequencies(limit, log_scale):
    view = frequency_data.head(int(limit)).copy()

    figure = px.bar(
        view,
        x="rank",
        y="frequency",
        color="cutoff_group",
        hover_name="object",
        hover_data={
            "rank": True,
            "frequency": True,
            "cutoff_group": False,
        },
        color_discrete_map={
            "Before cutoff": "#2563eb",
            "After cutoff": "#b8bec9",
        },
        title=f"Object frequency by rank: first {len(view):,} objects",
    )

    if calculated_cutoff_rank <= len(view):
        figure.add_vline(
            x=calculated_cutoff_rank + 0.5,
            line_width=2,
            line_dash="dash",
            line_color="#dc2626",
            annotation_text=f"Calculated cutoff after rank {calculated_cutoff_rank:,}",
            annotation_position="top right",
        )

    figure.update_layout(
        height=650,
        bargap=0,
        hovermode="closest",
        legend_title_text="Position",
        yaxis_type="log" if log_scale else "linear",
    )
    figure.update_xaxes(title="Object rank", rangeslider_visible=True)
    figure.update_yaxes(title="Frequency")
    figure.show()


limit_options = [
    ("Top 100", 100),
    ("Top 250", 250),
    ("Top 500", 500),
    ("Top 1,000", 1000),
    (f"All objects ({len(frequency_data):,})", len(frequency_data)),
]

controls = widgets.interactive(
    plot_object_frequencies,
    limit=widgets.Dropdown(
        options=limit_options,
        value=500,
        description="Display",
    ),
    log_scale=widgets.Checkbox(
        value=False,
        description="Log scale",
    ),
)
display(controls)

interactive(children=(Dropdown(description='Display', index=2, options=(('Top 100', 100), ('Top 250', 250), ('…

## First- and second-order relations

Relations are treated as connections in either direction. First-order objects connect directly to the analysis group. Second-order objects connect to a first-order object but are not already in the analysis group or first order.

In [5]:
import json
from collections import Counter, defaultdict

records_dir = Path("v1_1_objects_relations/records")
analysis_group = set(
    frequency_data.head(calculated_cutoff_rank)["object"].str.strip().str.lower()
)
adjacency = defaultdict(set)
pair_frequency = Counter()
analysis_group_relation_instances = 0

for record_path in records_dir.glob("*.json"):
    record = json.loads(record_path.read_text(encoding="utf-8-sig"))
    labels = {
        item["id"]: item["label"].strip().lower()
        for item in record.get("objects", [])
    }
    for relation in record.get("relations", []):
        subject = labels.get(relation.get("subject"))
        object_ = labels.get(relation.get("object"))
        if subject and object_ and subject != object_:
            pair_frequency[tuple(sorted((subject, object_)))] += 1
            adjacency[subject].add(object_)
            adjacency[object_].add(subject)
            if subject in analysis_group or object_ in analysis_group:
                analysis_group_relation_instances += 1

first_order = set().union(*(adjacency[item] for item in analysis_group)) - analysis_group
second_order = (
    set().union(*(adjacency[item] for item in first_order))
    - analysis_group
    - first_order
)

relation_order_summary = pd.DataFrame({
    "group": ["Analysis group", "First order", "Second order", "Total through second order"],
    "distinct_objects": [len(analysis_group), len(first_order), len(second_order),
                         len(analysis_group | first_order | second_order)],
})
display(relation_order_summary)
print(f"Relation instances touching the analysis group: {analysis_group_relation_instances:,}")

,group,distinct_objects
0,Analysis group,115
1,First order,3168
2,Second order,4059
3,Total through second order,7342


Relation instances touching the analysis group: 5,520


## Interactive relation network

Zoom or pan to inspect dense areas. Click a node to highlight its directly related objects and dim unrelated objects. Use **Reset selection** to restore the complete network.

In [6]:
import numpy as np
import plotly.graph_objects as go

all_network_nodes = analysis_group | first_order | second_order
visible_nodes = all_network_nodes.copy()
frequency_by_object = dict(zip(frequency_data["object"].str.strip().str.lower(),
                               frequency_data["frequency"]))
node_group = {node: "Analysis group" if node in analysis_group else
                       "First order" if node in first_order else "Second order"
              for node in all_network_nodes}

# Place analysis objects around an inner circle, then arrange their first- and
# second-order neighbourhoods in corresponding outer sectors.
analysis_nodes = sorted(analysis_group, key=lambda n: (-frequency_by_object.get(n, 0), n))
angles = {node: 2 * np.pi * index / len(analysis_nodes)
          for index, node in enumerate(analysis_nodes)}
positions = {node: (2 * np.cos(angles[node]), 2 * np.sin(angles[node]))
             for node in analysis_nodes}
first_anchor = {
    node: max(adjacency[node] & analysis_group,
              key=lambda n: (frequency_by_object.get(n, 0), n))
    for node in first_order
}
second_anchor = {}
for node in second_order:
    parent = max(adjacency[node] & first_order,
                 key=lambda n: (frequency_by_object.get(n, 0), n))
    second_anchor[node] = first_anchor[parent]

rng = np.random.default_rng(42)
sector_width = 2 * np.pi / len(analysis_nodes)
for nodes, anchors, inner_radius, outer_radius in [
    (sorted(first_order), first_anchor, 3.5, 7.0),
    (sorted(second_order), second_anchor, 7.5, 13.0),
]:
    for node in nodes:
        angle = angles[anchors[node]] + rng.uniform(-0.45, 0.45) * sector_width
        radius = rng.uniform(inner_radius, outer_radius)
        positions[node] = (radius * np.cos(angle), radius * np.sin(angle))

network_pair_frequency = {
    pair: count for pair, count in pair_frequency.items()
    if pair[0] in all_network_nodes and pair[1] in all_network_nodes
}

def network_at_threshold(threshold):
    pairs = [pair for pair, count in network_pair_frequency.items()
             if count >= threshold]
    nodes = analysis_group | {node for pair in pairs for node in pair}
    filtered_adjacency = defaultdict(set)
    for left, right in pairs:
        filtered_adjacency[left].add(right)
        filtered_adjacency[right].add(left)
    return nodes, sorted(pairs), filtered_adjacency

visible_nodes, edge_pairs, current_adjacency = network_at_threshold(1)
edge_x, edge_y = [], []
for left, right in edge_pairs:
    edge_x.extend([positions[left][0], positions[right][0], None])
    edge_y.extend([positions[left][1], positions[right][1], None])

base_edges = go.Scattergl(
    x=edge_x, y=edge_y, mode="lines", hoverinfo="skip",
    line=dict(width=0.35, color="rgba(100,116,139,0.025)"),
)
highlight_edges = go.Scattergl(
    x=[], y=[], mode="lines", hoverinfo="skip",
    line=dict(width=0.8, color="rgba(71,85,105,0.18)"),
)

draw_order = {"Second order": 0, "First order": 1, "Analysis group": 2}
group_colors = {"Analysis group": "#2563eb",
                "First order": "#10b981",
                "Second order": "#94a3b8"}

def node_visuals(nodes):
    labels = sorted(nodes, key=lambda n: (draw_order[node_group[n]],
                                           frequency_by_object.get(n, 0), n))
    colors = [group_colors[node_group[node]] for node in labels]
    opacity = [1.0 if node in analysis_group else 0.65 if node in first_order else 0.35
               for node in labels]
    frequencies = np.array([frequency_by_object.get(node, 1)
                            for node in labels], dtype=float)
    log_frequencies = np.log1p(frequencies)
    sizes = np.interp(log_frequencies, (np.log(2), np.log1p(frequency_data["frequency"].max())),
                      (3, 42)).tolist()
    return labels, colors, opacity, sizes

node_labels, base_colors, base_opacity, base_sizes = node_visuals(visible_nodes)
node_trace = go.Scattergl(
    x=[positions[node][0] for node in node_labels],
    y=[positions[node][1] for node in node_labels],
    mode="markers",
    customdata=[(node, node_group[node], frequency_by_object.get(node, 0))
                for node in node_labels],
    hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>Frequency: %{customdata[2]}<extra></extra>",
    marker=dict(color=base_colors, opacity=base_opacity, size=base_sizes,
                line=dict(width=0.7, color="white")),
)

network_figure = go.FigureWidget(data=[base_edges, highlight_edges, node_trace])
base_edges = network_figure.data[0]
highlight_edges = network_figure.data[1]
node_trace = network_figure.data[2]
network_figure.update_layout(
    title=f"Relation network: {len(visible_nodes):,} objects and {len(edge_pairs):,} object pairs",
    height=850, showlegend=False, hovermode="closest", dragmode="pan",
    margin=dict(l=10, r=10, t=55, b=10),
    plot_bgcolor="white",
    xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x", scaleratio=1),
)
network_figure._config = {"scrollZoom": True, "displaylogo": False}
selection_text = widgets.HTML("Click a node to highlight its direct relations.")
reset_button = widgets.Button(description="Reset selection", icon="refresh")
threshold_slider = widgets.IntSlider(
    value=1, min=1, max=4, step=1, description="Min pair frequency",
    continuous_update=False, readout=True, style={"description_width": "initial"},
)

def highlight_neighbours(trace, points, state):
    if not points.point_inds:
        return
    selected = node_labels[points.point_inds[0]]
    neighbours = current_adjacency[selected]
    colors = ["#dc2626" if node == selected else
              "#f59e0b" if node in neighbours else "#cbd5e1"
              for node in node_labels]
    opacity = [1.0 if node == selected else 0.9 if node in neighbours else 0.06
               for node in node_labels]
    sizes = [base_sizes[index] + 8 if node == selected else
             base_sizes[index] + 2 if node in neighbours else base_sizes[index]
             for index, node in enumerate(node_labels)]
    hx, hy = [], []
    for neighbour in neighbours:
        hx.extend([positions[selected][0], positions[neighbour][0], None])
        hy.extend([positions[selected][1], positions[neighbour][1], None])
    with network_figure.batch_update():
        node_trace.marker.color = colors
        node_trace.marker.opacity = opacity
        node_trace.marker.size = sizes
        highlight_edges.x = hx
        highlight_edges.y = hy
    selection_text.value = (f"<b>{selected}</b>: {len(neighbours):,} directly related objects")

def reset_selection(_):
    with network_figure.batch_update():
        node_trace.marker.color = base_colors
        node_trace.marker.opacity = base_opacity
        node_trace.marker.size = base_sizes
        highlight_edges.x = []
        highlight_edges.y = []
    selection_text.value = "Click a node to highlight its direct relations."

def change_threshold(change):
    global visible_nodes, edge_pairs, current_adjacency
    global node_labels, base_colors, base_opacity, base_sizes
    visible_nodes, edge_pairs, current_adjacency = network_at_threshold(change["new"])
    node_labels, base_colors, base_opacity, base_sizes = node_visuals(visible_nodes)
    edge_x, edge_y = [], []
    for left, right in edge_pairs:
        edge_x.extend([positions[left][0], positions[right][0], None])
        edge_y.extend([positions[left][1], positions[right][1], None])
    with network_figure.batch_update():
        base_edges.x = edge_x
        base_edges.y = edge_y
        highlight_edges.x = []
        highlight_edges.y = []
        node_trace.x = [positions[node][0] for node in node_labels]
        node_trace.y = [positions[node][1] for node in node_labels]
        node_trace.customdata = [(node, node_group[node], frequency_by_object.get(node, 0))
                                 for node in node_labels]
        node_trace.marker.color = base_colors
        node_trace.marker.opacity = base_opacity
        node_trace.marker.size = base_sizes
        network_figure.layout.title = (
            f"Relation network (pair frequency ≥ {change['new']}): "
            f"{len(visible_nodes):,} objects and {len(edge_pairs):,} object pairs"
        )
        network_figure.layout.xaxis.autorange = True
        network_figure.layout.yaxis.autorange = True
    selection_text.value = "Click a node to highlight its direct relations."

node_trace.on_click(highlight_neighbours)
reset_button.on_click(reset_selection)
threshold_slider.observe(change_threshold, names="value")
display(widgets.VBox([widgets.HBox([threshold_slider, reset_button, selection_text]),
                       network_figure]))

## Exercise-to-exercise relation groups

This section keeps direct relations where both endpoints are exercises, counts each unordered exercise pair, applies a minimum pair-frequency threshold, and finds connected components. The structural grouping is automatic; interpreting and naming the structures within a component remains a semantic judgement.

In [7]:
exercise_names = {
    "back squat", "bear-hug carry", "bench press", "bodyweight bench press",
    "bodyweight clean and press", "bulgarian goat-bag swing", "clean and jerk",
    "clean and press", "dead hang", "deadlift", "double-bodyweight deadlift",
    "double-kettlebell clean", "double-kettlebell front squat", "double-kettlebell press",
    "farmer walk", "front squat", "glute bridge", "goblet squat",
    "half-kneeling press", "high-repetition back squat", "high-repetition press",
    "high-repetition squat", "hip thrust", "humane burpee",
    "kettlebell clean and press", "kettlebell snatch", "kettlebell swing",
    "military press", "moving plank", "naked get-up", "neck nod",
    "overhead press", "overhead squat", "post-workout walk", "power clean",
    "pull-up", "push press", "push-up", "sandbag carry", "six-point rock",
    "strict press", "suitcase carry", "swimming", "t-y-i drill",
    "turkish get-up", "waiter walk", "walking", "whip snatch",
}

exercise_pair_frequency = Counter()
exercise_pair_relations = defaultdict(Counter)
exercise_pair_transcripts = defaultdict(set)

for record_path in records_dir.glob("*.json"):
    record = json.loads(record_path.read_text(encoding="utf-8-sig"))
    labels = {item["id"]: item["label"].strip().lower()
              for item in record.get("objects", [])}
    for relation in record.get("relations", []):
        subject = labels.get(relation.get("subject"))
        object_ = labels.get(relation.get("object"))
        if subject in exercise_names and object_ in exercise_names and subject != object_:
            pair = tuple(sorted((subject, object_)))
            exercise_pair_frequency[pair] += 1
            exercise_pair_relations[pair][relation.get("relation", "")] += 1
            exercise_pair_transcripts[pair].add(record.get("video_id", record_path.stem))

def connected_exercise_groups(pair_counts, minimum_frequency):
    graph = defaultdict(set)
    for (left, right), count in pair_counts.items():
        if count >= minimum_frequency:
            graph[left].add(right)
            graph[right].add(left)
    groups, visited = [], set()
    for start in sorted(graph):
        if start in visited:
            continue
        stack, component = [start], set()
        visited.add(start)
        while stack:
            node = stack.pop()
            component.add(node)
            for neighbour in graph[node] - visited:
                visited.add(neighbour)
                stack.append(neighbour)
        groups.append(sorted(component))
    return sorted(groups, key=lambda group: (-len(group), group))

minimum_exercise_pair_frequency = 3
strong_exercise_pairs = {pair: count for pair, count in exercise_pair_frequency.items()
                         if count >= minimum_exercise_pair_frequency}
exercise_groups = connected_exercise_groups(
    exercise_pair_frequency, minimum_exercise_pair_frequency
)

pair_rows = [
    {"exercise_1": pair[0], "exercise_2": pair[1], "pair_frequency": count,
     "transcripts": len(exercise_pair_transcripts[pair]),
     "relation_types": ", ".join(
         f"{name} ({frequency})" for name, frequency in
         exercise_pair_relations[pair].most_common())}
    for pair, count in sorted(strong_exercise_pairs.items(),
                              key=lambda item: (-item[1], item[0]))
]
group_rows = [{"group": index, "exercise_count": len(group),
               "exercises": "; ".join(group)}
              for index, group in enumerate(exercise_groups, start=1)]

print(f"Exercises listed: {len(exercise_names):,}")
print(f"Direct exercise relation instances: {sum(exercise_pair_frequency.values()):,}")
print(f"Distinct exercise pairs: {len(exercise_pair_frequency):,}")
print(f"At pair frequency ≥ {minimum_exercise_pair_frequency}: "
      f"{len(strong_exercise_pairs):,} pairs forming {len(exercise_groups):,} groups")
display(pd.DataFrame(pair_rows))
display(pd.DataFrame(group_rows))

Exercises listed: 48
Direct exercise relation instances: 124
Distinct exercise pairs: 58
At pair frequency ≥ 3: 14 pairs forming 5 groups


,exercise_1,exercise_2,pair_frequency,transcripts,relation_types
0,back squat,front squat,9,8,"precedes (3), maintained capacity for (1), dif..."
1,humane burpee,push-up,8,8,"combines (5), includes (2), is structurally im..."
2,bulgarian goat-bag swing,kettlebell swing,6,6,"regresses (2), can replace (2), prepares for (..."
3,goblet squat,humane burpee,6,6,"combines (4), includes (2)"
4,goblet squat,overhead squat,6,6,"precedes (3), can transition to (1), can follo..."
5,hip thrust,kettlebell swing,6,5,"can replace (2), teaches position for (1), pre..."
6,humane burpee,kettlebell swing,6,6,"combines (4), includes (2)"
7,front squat,overhead squat,5,5,"precedes (3), precedes the (1), progresses to (1)"
8,kettlebell snatch,kettlebell swing,4,4,"shares qualities with (1), contains (1), can a..."
9,push press,strict press,4,4,"can precede (2), can progress load beyond (1),..."


,group,exercise_count,exercises
0,1,11,back squat; bulgarian goat-bag swing; front sq...
1,2,2,clean and jerk; clean and press
2,3,2,moving plank; suitcase carry
3,4,2,naked get-up; turkish get-up
4,5,2,push press; strict press


# Clean-slate searchable dashboard

This section leaves the experimental workflow above intact. It independently reloads the saved object-frequency table and per-video relation records, rebuilds the indexes, and presents the test dashboard.

Use **Find object** to type any part of a name. The ranked suggestions show matching objects; selecting one redraws the network as that object's direct or second-order neighbourhood. The object scope, minimum pair frequency, and depth controls recalculate the graph. The table beneath the graph shows every directed `subject → relation → object` statement without truncating rows or text.

In [8]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import html
import json
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


# Clean-slate data preparation: this section deliberately rebuilds its own
# indexes from the saved CSV and per-video JSON records.
dashboard_frequency_path = Path("object_frequencies.csv")
dashboard_records_dir = Path("v1_1_objects_relations/records")

dashboard_frequency_data = (
    pd.read_csv(dashboard_frequency_path)
    .sort_values("rank")
    .reset_index(drop=True)
)
dashboard_frequency_data["object_key"] = (
    dashboard_frequency_data["object"].astype(str).str.strip().str.lower()
)

dashboard_frequency_by_object = dict(zip(
    dashboard_frequency_data["object_key"],
    dashboard_frequency_data["frequency"],
))
dashboard_rank_by_object = dict(zip(
    dashboard_frequency_data["object_key"],
    dashboard_frequency_data["rank"],
))

# Recalculate the recurring-object elbow used by the earlier workflow.
dashboard_recurring = dashboard_frequency_data.loc[
    dashboard_frequency_data["frequency"] > 1
].copy()
dashboard_normalized_rank = (
    (dashboard_recurring["rank"] - dashboard_recurring["rank"].min())
    / (dashboard_recurring["rank"].max() - dashboard_recurring["rank"].min())
)
dashboard_normalized_frequency = (
    (dashboard_recurring["frequency"] - dashboard_recurring["frequency"].min())
    / (dashboard_recurring["frequency"].max() - dashboard_recurring["frequency"].min())
)
dashboard_knee_index = (
    (1 - dashboard_normalized_rank) - dashboard_normalized_frequency
).idxmax()
dashboard_cutoff_frequency = int(
    dashboard_frequency_data.loc[dashboard_knee_index, "frequency"]
)
dashboard_cutoff_rank = int(
    dashboard_frequency_data.loc[
        dashboard_frequency_data["frequency"] >= dashboard_cutoff_frequency,
        "rank",
    ].max()
)

dashboard_analysis_group = set(
    dashboard_frequency_data.head(dashboard_cutoff_rank)["object_key"]
)
dashboard_pair_frequency = Counter()
dashboard_triple_frequency = Counter()
dashboard_triple_videos = defaultdict(set)
dashboard_full_adjacency = defaultdict(set)

for dashboard_record_path in dashboard_records_dir.glob("*.json"):
    dashboard_record = json.loads(
        dashboard_record_path.read_text(encoding="utf-8-sig")
    )
    dashboard_labels = {
        item["id"]: item["label"].strip().lower()
        for item in dashboard_record.get("objects", [])
    }
    for dashboard_relation in dashboard_record.get("relations", []):
        dashboard_subject = dashboard_labels.get(dashboard_relation.get("subject"))
        dashboard_object = dashboard_labels.get(dashboard_relation.get("object"))
        if not dashboard_subject or not dashboard_object or dashboard_subject == dashboard_object:
            continue
        dashboard_pair = tuple(sorted((dashboard_subject, dashboard_object)))
        dashboard_relation_text = str(
            dashboard_relation.get("relation", "unspecified")
        ).strip()
        dashboard_triple = (
            dashboard_subject, dashboard_relation_text, dashboard_object
        )
        dashboard_video_id = dashboard_record.get(
            "video_id", dashboard_record_path.stem
        )
        dashboard_pair_frequency[dashboard_pair] += 1
        dashboard_triple_frequency[dashboard_triple] += 1
        dashboard_triple_videos[dashboard_triple].add(dashboard_video_id)
        dashboard_full_adjacency[dashboard_subject].add(dashboard_object)
        dashboard_full_adjacency[dashboard_object].add(dashboard_subject)

dashboard_first_order = (
    set().union(*(
        dashboard_full_adjacency[node]
        for node in dashboard_analysis_group
    )) - dashboard_analysis_group
)
dashboard_second_order = (
    set().union(*(
        dashboard_full_adjacency[node]
        for node in dashboard_first_order
    ))
    - dashboard_analysis_group
    - dashboard_first_order
)
dashboard_relation_nodes = (
    dashboard_analysis_group | dashboard_first_order | dashboard_second_order
)

dashboard_exercise_names = {
    "back squat", "bear-hug carry", "bench press", "bodyweight bench press",
    "bodyweight clean and press", "bulgarian goat-bag swing", "clean and jerk",
    "clean and press", "dead hang", "deadlift", "double-bodyweight deadlift",
    "double-kettlebell clean", "double-kettlebell front squat",
    "double-kettlebell press", "farmer walk", "front squat", "glute bridge",
    "goblet squat", "half-kneeling press", "high-repetition back squat",
    "high-repetition press", "high-repetition squat", "hip thrust",
    "humane burpee", "kettlebell clean and press", "kettlebell snatch",
    "kettlebell swing", "military press", "moving plank", "naked get-up",
    "neck nod", "overhead press", "overhead squat", "post-workout walk",
    "power clean", "pull-up", "push press", "push-up", "sandbag carry",
    "six-point rock", "strict press", "suitcase carry", "swimming",
    "t-y-i drill", "turkish get-up", "waiter walk", "walking", "whip snatch",
}


def dashboard_node_group(node):
    if node in dashboard_analysis_group:
        return "Analysis group"
    if node in dashboard_first_order:
        return "First order"
    if node in dashboard_second_order:
        return "Second order"
    return "Outside relation network"


dashboard_scope_nodes = {
    "all": set(dashboard_frequency_data["object_key"]),
    "network": set(dashboard_relation_nodes),
    "analysis": set(dashboard_analysis_group),
    "analysis_first": set(dashboard_analysis_group | dashboard_first_order),
    "exercises": set(dashboard_exercise_names & set(dashboard_frequency_by_object)),
}

# Stable full-network positions. Local neighbourhoods receive a fresh radial
# layout centred on the selected object.
dashboard_analysis_nodes = sorted(
    dashboard_analysis_group,
    key=lambda node: (-dashboard_frequency_by_object.get(node, 0), node),
)
dashboard_angles = {
    node: 2 * np.pi * index / len(dashboard_analysis_nodes)
    for index, node in enumerate(dashboard_analysis_nodes)
}
dashboard_positions = {
    node: (2 * np.cos(dashboard_angles[node]), 2 * np.sin(dashboard_angles[node]))
    for node in dashboard_analysis_nodes
}
dashboard_first_anchor = {
    node: max(
        dashboard_full_adjacency[node] & dashboard_analysis_group,
        key=lambda neighbour: (
            dashboard_frequency_by_object.get(neighbour, 0), neighbour
        ),
    )
    for node in dashboard_first_order
}
dashboard_second_anchor = {}
for dashboard_node in dashboard_second_order:
    dashboard_parents = dashboard_full_adjacency[dashboard_node] & dashboard_first_order
    if dashboard_parents:
        dashboard_parent = max(
            dashboard_parents,
            key=lambda neighbour: (
                dashboard_frequency_by_object.get(neighbour, 0), neighbour
            ),
        )
        dashboard_second_anchor[dashboard_node] = dashboard_first_anchor[dashboard_parent]

dashboard_rng = np.random.default_rng(42)
dashboard_sector_width = 2 * np.pi / len(dashboard_analysis_nodes)
for dashboard_nodes, dashboard_anchors, dashboard_inner, dashboard_outer in [
    (sorted(dashboard_first_order), dashboard_first_anchor, 3.5, 7.0),
    (sorted(dashboard_second_anchor), dashboard_second_anchor, 7.5, 13.0),
]:
    for dashboard_node in dashboard_nodes:
        dashboard_angle = (
            dashboard_angles[dashboard_anchors[dashboard_node]]
            + dashboard_rng.uniform(-0.45, 0.45) * dashboard_sector_width
        )
        dashboard_radius = dashboard_rng.uniform(dashboard_inner, dashboard_outer)
        dashboard_positions[dashboard_node] = (
            dashboard_radius * np.cos(dashboard_angle),
            dashboard_radius * np.sin(dashboard_angle),
        )

print(
    f"Clean dashboard index: {len(dashboard_frequency_data):,} objects, "
    f"{len(dashboard_pair_frequency):,} relation pairs, "
    f"cutoff rank {dashboard_cutoff_rank:,}."
)


# Dashboard controls.
dashboard_search = widgets.Text(
    description="Find object",
    placeholder="Type part of an object name...",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "90px"},
)
dashboard_matches = widgets.Select(
    options=[],
    rows=8,
    description="Matches",
    layout=widgets.Layout(width="100%", height="180px"),
    style={"description_width": "90px"},
)
dashboard_scope = widgets.Dropdown(
    options=[
        ("All extracted objects", "all"),
        ("Relation-network objects", "network"),
        ("Analysis group", "analysis"),
        ("Analysis + first order", "analysis_first"),
        ("Exercises only", "exercises"),
    ],
    value="network",
    description="Object scope",
    style={"description_width": "initial"},
)
dashboard_min_pair = widgets.IntSlider(
    value=3,
    min=1,
    max=4,
    step=1,
    description="Min pair frequency",
    continuous_update=False,
    style={"description_width": "initial"},
)
dashboard_depth = widgets.ToggleButtons(
    options=[("Direct", 1), ("Second order", 2)],
    value=1,
    description="Depth",
    style={"description_width": "initial"},
)
dashboard_reset = widgets.Button(description="Reset selection", icon="refresh")
dashboard_search_status = widgets.HTML()
dashboard_selected_summary = widgets.HTML()
dashboard_match_table = widgets.Output()
dashboard_relation_table = widgets.Output()

dashboard_edge_trace = go.Scattergl(
    x=[], y=[], mode="lines", hoverinfo="skip",
    line=dict(width=0.45, color="rgba(100,116,139,0.08)"),
)
dashboard_selected_edge_trace = go.Scattergl(
    x=[], y=[], mode="lines", hoverinfo="skip",
    line=dict(width=1.4, color="rgba(245,158,11,0.60)"),
)
dashboard_node_trace = go.Scattergl(
    x=[], y=[], mode="markers", customdata=[],
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>%{customdata[1]}"
        "<br>Frequency: %{customdata[2]}<br>Rank: %{customdata[3]}<extra></extra>"
    ),
    marker=dict(line=dict(width=0.7, color="white")),
)
dashboard_figure = go.FigureWidget(data=[
    dashboard_edge_trace,
    dashboard_selected_edge_trace,
    dashboard_node_trace,
])
dashboard_edge_trace = dashboard_figure.data[0]
dashboard_selected_edge_trace = dashboard_figure.data[1]
dashboard_node_trace = dashboard_figure.data[2]
dashboard_figure.update_layout(
    title="Searchable relation network",
    height=780,
    showlegend=False,
    hovermode="closest",
    dragmode="pan",
    margin=dict(l=10, r=10, t=55, b=10),
    plot_bgcolor="white",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, scaleanchor="x", scaleratio=1),
)
dashboard_figure._config = {
    "scrollZoom": True,
    "displaylogo": False,
    "responsive": True,
}

dashboard_state = {
    "selected": None,
    "node_labels": [],
    "visible_direct": set(),
}
dashboard_max_local_nodes = 250
dashboard_group_colors = {
    "Analysis group": "#2563eb",
    "First order": "#10b981",
    "Second order": "#94a3b8",
    "Outside relation network": "#cbd5e1",
}


def dashboard_filtered_pairs():
    allowed_nodes = dashboard_scope_nodes[dashboard_scope.value]
    return {
        pair: count
        for pair, count in dashboard_pair_frequency.items()
        if count >= dashboard_min_pair.value
        and pair[0] in allowed_nodes
        and pair[1] in allowed_nodes
    }


def dashboard_build_adjacency(filtered_pairs):
    filtered_adjacency = defaultdict(set)
    for left, right in filtered_pairs:
        filtered_adjacency[left].add(right)
        filtered_adjacency[right].add(left)
    return filtered_adjacency


def dashboard_search_key(node, query):
    if not query:
        return (4, -dashboard_frequency_by_object.get(node, 0), node)
    if node == query:
        match_quality = 0
    elif node.startswith(query):
        match_quality = 1
    elif re.search(rf"(?:^|[\s\-]){re.escape(query)}", node):
        match_quality = 2
    else:
        match_quality = 3
    return (match_quality, -dashboard_frequency_by_object.get(node, 0), node)


def dashboard_update_matches(*_):
    query = dashboard_search.value.strip().lower()
    candidates = dashboard_scope_nodes[dashboard_scope.value]
    if query:
        candidates = {node for node in candidates if query in node}
    ranked = sorted(
        candidates,
        key=lambda node: dashboard_search_key(node, query),
    )[:25]
    dashboard_matches.options = [
        (
            f"#{int(dashboard_rank_by_object.get(node, 0)):,}  {node}  "
            f"(frequency {int(dashboard_frequency_by_object.get(node, 0)):,})",
            node,
        )
        for node in ranked
    ]
    dashboard_search_status.value = (
        f"<b>{len(candidates):,}</b> text matches; showing the first "
        f"<b>{len(ranked):,}</b>."
        if query else
        f"Showing the <b>{len(ranked):,}</b> most frequent objects in this scope."
    )
    match_rows = [
        {
            "rank": int(dashboard_rank_by_object.get(node, 0)),
            "object": node,
            "frequency": int(dashboard_frequency_by_object.get(node, 0)),
            "network_group": dashboard_node_group(node),
            "direct_connections": len(dashboard_full_adjacency[node]),
        }
        for node in ranked
    ]
    with dashboard_match_table:
        dashboard_match_table.clear_output(wait=True)
        display(pd.DataFrame(match_rows))


def dashboard_edge_coordinates(pairs, position_map):
    edge_x, edge_y = [], []
    for left, right in pairs:
        if left not in position_map or right not in position_map:
            continue
        edge_x.extend([position_map[left][0], position_map[right][0], None])
        edge_y.extend([position_map[left][1], position_map[right][1], None])
    return edge_x, edge_y


def dashboard_local_positions(selected, direct_nodes, second_nodes):
    local_positions = {selected: (0.0, 0.0)}
    direct_sorted = sorted(
        direct_nodes,
        key=lambda node: (
            -dashboard_pair_frequency[tuple(sorted((selected, node)))],
            -dashboard_frequency_by_object.get(node, 0),
            node,
        ),
    )
    for index, node in enumerate(direct_sorted):
        angle = 2 * np.pi * index / max(1, len(direct_sorted))
        local_positions[node] = (3.2 * np.cos(angle), 3.2 * np.sin(angle))
    second_sorted = sorted(
        second_nodes,
        key=lambda node: (-dashboard_frequency_by_object.get(node, 0), node),
    )
    for index, node in enumerate(second_sorted):
        angle = 2 * np.pi * index / max(1, len(second_sorted))
        local_positions[node] = (6.2 * np.cos(angle), 6.2 * np.sin(angle))
    return local_positions


def dashboard_limit_neighbourhood(selected, adjacency):
    direct = set(adjacency[selected])
    direct_ranked = sorted(
        direct,
        key=lambda node: (
            -dashboard_pair_frequency[tuple(sorted((selected, node)))],
            -dashboard_frequency_by_object.get(node, 0),
            node,
        ),
    )
    kept_direct = set(direct_ranked[: dashboard_max_local_nodes - 1])
    kept_second = set()
    if dashboard_depth.value == 2 and len(kept_direct) < dashboard_max_local_nodes - 1:
        second = set().union(*(adjacency[node] for node in kept_direct))
        second -= kept_direct | {selected}
        remaining = dashboard_max_local_nodes - 1 - len(kept_direct)
        kept_second = set(sorted(
            second,
            key=lambda node: (-dashboard_frequency_by_object.get(node, 0), node),
        )[:remaining])
    return direct, kept_direct, kept_second


def dashboard_node_sizes(nodes, selected=None):
    max_frequency = max(dashboard_frequency_by_object.values())
    sizes = []
    for node in nodes:
        frequency = max(1, dashboard_frequency_by_object.get(node, 1))
        size = float(np.interp(
            np.log1p(frequency),
            (np.log(2), np.log1p(max_frequency)),
            (5, 38),
        ))
        if node == selected:
            size += 10
        sizes.append(size)
    return sizes


def dashboard_render_relation_table(selected, adjacency, visible_direct):
    visible_pairs = {
        tuple(sorted((selected, neighbour)))
        for neighbour in visible_direct
    }
    rows = []
    for triple, frequency in dashboard_triple_frequency.items():
        subject, relation, object_ = triple
        if tuple(sorted((subject, object_))) not in visible_pairs:
            continue
        rows.append({
            "subject": subject,
            "relation": relation,
            "object": object_,
            "frequency": frequency,
            "supporting_videos": len(dashboard_triple_videos[triple]),
        })

    with dashboard_relation_table:
        dashboard_relation_table.clear_output(wait=True)
        if rows:
            relation_frame = pd.DataFrame(rows)
            relation_frame["subject_total"] = relation_frame.groupby(
                "subject"
            )["frequency"].transform("sum")
            relation_frame = (
                relation_frame
                .sort_values(
                    ["subject_total", "subject", "frequency",
                     "relation", "object"],
                    ascending=[False, True, False, True, True],
                )
                .drop(columns="subject_total")
                .set_index(["subject", "relation", "object"])
            )
        else:
            relation_frame = pd.DataFrame(
                columns=["subject", "relation", "object",
                         "frequency", "supporting_videos"]
            ).set_index(["subject", "relation", "object"])

        with pd.option_context(
            "display.max_rows", None,
            "display.max_columns", None,
            "display.max_colwidth", None,
            "display.multi_sparse", True,
        ):
            display(relation_frame)


def dashboard_render_network(*_):
    selected = dashboard_state["selected"]
    filtered_pairs = dashboard_filtered_pairs()
    filtered_adjacency = dashboard_build_adjacency(filtered_pairs)
    scope_nodes = dashboard_scope_nodes[dashboard_scope.value]

    if selected not in scope_nodes:
        selected = None
        dashboard_state["selected"] = None

    if selected:
        all_direct, direct_nodes, second_nodes = dashboard_limit_neighbourhood(
            selected, filtered_adjacency
        )
        visible_nodes = {selected} | direct_nodes | second_nodes
        visible_pairs = {
            pair: count
            for pair, count in filtered_pairs.items()
            if pair[0] in visible_nodes and pair[1] in visible_nodes
        }
        direct_pairs = {
            pair: count
            for pair, count in visible_pairs.items()
            if selected in pair
        }
        base_pairs = {
            pair: count
            for pair, count in visible_pairs.items()
            if selected not in pair
        }
        position_map = dashboard_local_positions(
            selected, direct_nodes, second_nodes
        )
        node_labels = [selected] + sorted(
            direct_nodes,
            key=lambda node: (
                -dashboard_pair_frequency[tuple(sorted((selected, node)))], node
            ),
        ) + sorted(second_nodes)
        node_colors = [
            "#dc2626" if node == selected else
            "#f59e0b" if node in direct_nodes else "#14b8a6"
            for node in node_labels
        ]
        node_opacity = [1.0] * len(node_labels)
        omitted = max(0, len(all_direct) - len(direct_nodes))
        dashboard_selected_summary.value = (
            f"<b>{html.escape(selected)}</b> &mdash; rank "
            f"{int(dashboard_rank_by_object.get(selected, 0)):,}, frequency "
            f"{int(dashboard_frequency_by_object.get(selected, 0)):,}, "
            f"{len(all_direct):,} direct relations under the active filters."
            + (
                f" Display limited to the strongest {len(direct_nodes):,}; "
                f"{omitted:,} omitted."
                if omitted else ""
            )
        )
        dashboard_render_relation_table(selected, filtered_adjacency, direct_nodes)
        title = (
            f"Neighbourhood of {selected}: {len(visible_nodes):,} objects and "
            f"{len(visible_pairs):,} pairs"
        )
    else:
        visible_pairs = filtered_pairs
        if dashboard_scope.value in {"all", "network", "analysis_first"}:
            visible_nodes = dashboard_analysis_group | {
                node for pair in visible_pairs for node in pair
            }
        else:
            visible_nodes = {
                node for pair in visible_pairs for node in pair
            }
            if dashboard_scope.value == "analysis":
                visible_nodes |= dashboard_analysis_group
            elif dashboard_scope.value == "exercises":
                visible_nodes |= dashboard_exercise_names & scope_nodes
        visible_nodes &= scope_nodes
        node_labels = sorted(
            visible_nodes,
            key=lambda node: (
                {"Second order": 0, "First order": 1, "Analysis group": 2,
                 "Outside relation network": -1}[dashboard_node_group(node)],
                dashboard_frequency_by_object.get(node, 0),
                node,
            ),
        )
        position_map = {
            node: dashboard_positions[node]
            for node in node_labels
            if node in dashboard_positions
        }
        node_labels = [node for node in node_labels if node in position_map]
        node_colors = [
            dashboard_group_colors[dashboard_node_group(node)]
            for node in node_labels
        ]
        node_opacity = [
            1.0 if node in dashboard_analysis_group else
            0.65 if node in dashboard_first_order else 0.35
            for node in node_labels
        ]
        base_pairs = visible_pairs
        direct_pairs = {}
        dashboard_selected_summary.value = (
            "No object selected. Search above or click a node to isolate its "
            "relational neighbourhood."
        )
        with dashboard_relation_table:
            dashboard_relation_table.clear_output(wait=True)
        title = (
            f"Filtered relation network: {len(node_labels):,} objects and "
            f"{len(visible_pairs):,} pairs"
        )

    base_x, base_y = dashboard_edge_coordinates(base_pairs, position_map)
    selected_x, selected_y = dashboard_edge_coordinates(direct_pairs, position_map)
    dashboard_state["node_labels"] = node_labels
    dashboard_state["visible_direct"] = set(
        filtered_adjacency[selected]
    ) if selected else set()

    with dashboard_figure.batch_update():
        dashboard_edge_trace.x = base_x
        dashboard_edge_trace.y = base_y
        dashboard_selected_edge_trace.x = selected_x
        dashboard_selected_edge_trace.y = selected_y
        dashboard_node_trace.x = [position_map[node][0] for node in node_labels]
        dashboard_node_trace.y = [position_map[node][1] for node in node_labels]
        dashboard_node_trace.customdata = [
            (
                node,
                dashboard_node_group(node),
                int(dashboard_frequency_by_object.get(node, 0)),
                int(dashboard_rank_by_object.get(node, 0)),
            )
            for node in node_labels
        ]
        dashboard_node_trace.marker.color = node_colors
        dashboard_node_trace.marker.opacity = node_opacity
        dashboard_node_trace.marker.size = dashboard_node_sizes(
            node_labels, selected
        )
        dashboard_figure.layout.title = title
        dashboard_figure.layout.xaxis.autorange = True
        dashboard_figure.layout.yaxis.autorange = True


def dashboard_select_object(node):
    if not node:
        return
    dashboard_state["selected"] = node
    if dashboard_search.value != node:
        dashboard_search.value = node
    dashboard_render_network()


def dashboard_on_match(change):
    if change.get("name") == "value" and change.get("new"):
        dashboard_select_object(change["new"])


def dashboard_on_node_click(trace, points, state):
    if not points.point_inds:
        return
    dashboard_select_object(
        dashboard_state["node_labels"][points.point_inds[0]]
    )


def dashboard_on_filter(change):
    if change.get("name") != "value":
        return
    dashboard_update_matches()
    dashboard_render_network()


def dashboard_reset_selection(_):
    dashboard_state["selected"] = None
    dashboard_search.value = ""
    dashboard_update_matches()
    dashboard_render_network()


dashboard_search.observe(dashboard_update_matches, names="value")
dashboard_matches.observe(dashboard_on_match, names="value")
dashboard_scope.observe(dashboard_on_filter, names="value")
dashboard_min_pair.observe(dashboard_on_filter, names="value")
dashboard_depth.observe(dashboard_on_filter, names="value")
dashboard_reset.on_click(dashboard_reset_selection)
dashboard_node_trace.on_click(dashboard_on_node_click)

dashboard_update_matches()
dashboard_render_network()

dashboard_controls = widgets.VBox([
    dashboard_search,
    dashboard_search_status,
    dashboard_matches,
    dashboard_match_table,
    widgets.HBox([dashboard_scope, dashboard_min_pair]),
    widgets.HBox([dashboard_depth, dashboard_reset]),
    dashboard_selected_summary,
])

display(widgets.VBox([
    dashboard_controls,
    dashboard_figure,
    widgets.HTML("<h4>Relations for the selected object</h4>"),
    dashboard_relation_table,
]))


Clean dashboard index: 18,254 objects, 20,198 relation pairs, cutoff rank 115.
